# Bestellungen auswerten

TODO: Verschieb die Funktionen nach `analysis.py` und importiere sie hier. Lösche danach die Zellen mit `def`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
def load_items(path):
    orders = json.loads(path.read_text(encoding="utf-8"))
    items = pd.json_normalize(orders, record_path="items", meta=["id", "customerId", "status", "createdAt"])
    items = items.rename(columns={"id": "orderId"})
    items["createdAt"] = pd.to_datetime(items["createdAt"])
    items["revenueCents"] = items["quantity"] * items["priceCents"]
    return items

In [ ]:
items = load_items(Path("../../../data/orders.json"))
items.head()

## Umsatz je Produkt in Euro

In [ ]:
done = items[~items["status"].isin(["cancelled", "draft"])]
(done.groupby("productId")["revenueCents"].sum().sort_values(ascending=False) / 100).round(2)

## Umsatz je Monat in Euro

In [ ]:
# TODO: Diese Zelle nutzt `done` aus einer früheren Zelle – nach „Restart & Run All“ ist das fragil.
(done.groupby(done["createdAt"].dt.strftime("%Y-%m"))["revenueCents"].sum() / 100).round(2)